Evaluation of NicheNet’s ligand-target predictions
================

This notebook shows how the ligand-target predictions of NicheNet were
evaluated. For validation, we collected transcriptome data of cells
before and after they were treated by one or two ligands in culture.
Using these ligand treatment datasets for validation has the advantage
that observed gene expression changes can be directly attributed to the
addition of the ligand(s). Hence, differentially expressed genes can be
considered as a gold standard of target genes of a particular ligand.

In [1]:
from nichenetpy.utils import read_csv_cols, read_csv_rows
from nichenetpy.model_construction import (
    construct_weighted_networks,
    construct_ligand_target_matrix,
    apply_hub_correction
)
from nichenetpy.evaluation import convert_expression_settings_evaluation

from itertools import chain, repeat

import os
import requests
import pandas as pd
import json
import pickle

In [2]:
data_path = os.path.normpath("./tutorial_files/model_evaluation")
if not os.path.exists(data_path):
    os.makedirs(data_path)
for filename in (
    "cytosig_settings.json",
    "expression_settings_validation.json"
):
    file_path = os.path.join(data_path, filename)
    if not os.path.exists(file_path):
        res = requests.get(f"https://zenodo.org/records/15228527/files/{filename}")
        with open(file_path, "wb") as file:
            file.write(res.content)
filename = "nichenet_human.pkl"
file_path = os.path.join("./tutorial_files", filename)
if not os.path.exists(file_path):
    res = requests.get(f"https://zenodo.org/records/14887637/files/{filename}")
    with open(file_path, "wb") as file:
        file.write(res.content)

### Example: transcriptional response prediction evaluation

First, we will demonstrate how to evaluate the transcriptional response
(i.e. target gene prediction) performance for all ligand treatment
expression datasets. For this, we determine how well the model predicts
which genes are differentially expressed after treatment with a ligand.
Ideally, target genes with high regulatory potential scores for a
ligand, should be differentially expressed in response to that ligand.

For information of all collected ligand treatment datasets, see [Dataset
information](https://github.com/saeyslab/nichenetr/blob/master/vignettes/evaluation_datasets.xlsx)

For the sake of simplicity, we exclude in this vignette the
ligand-treatment datasets profiling the response to multiple ligands. To
see how to build a ligand-target model with target predictions for
multiple ligands at once: see [Construction of NicheNet’s
ligand-target model](model_construction.ipynb).

In [3]:
with open(os.path.join("./tutorial_files", "nichenet_human.pkl"), "rb") as file:
    model = pickle.loads(file.read())
predictor = model["predictor"]

Step 1: convert expression datasets to the required format to perform target gene prediction

In [4]:
with open(os.path.join(data_path, "expression_settings_validation.json"), "rb") as file:
    settings = json.loads(file.read())
settings = {
    k: convert_expression_settings_evaluation(v)
    for k, v in settings.items()
    if type(v["from"]) is str or len(v["from"]) == 1
}

Step 2: calculate the target gene prediction performances

In [6]:
performances = {
    k: predictor.evaluate_target_prediction(v["from"], v["response"])
    for k, v in settings.items()
}

Step 3: visualize the results: show different classification evaluation metrics